In [1]:
# Import standard libraries
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

In [2]:
# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import the custom environment and PPO training module
from envs.balance_bot_env import BalanceBotEnv
from rl.ppo_trainer import train, PPOConfig

In [3]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 1   # Single environment for now (so we can watch it train)

In [4]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    total_timesteps = 3_500,     # Total simulation steps across all envs and iterations
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.002,              # Match MJCF opt.timestep for real-time rendering
)

In [5]:
# Build the environment
env = BalanceBotEnv(
    mjcf_path    = MJCF_PATH,
    render_mode  = "human",
)

# Wrap in RecordEpisodeStatistics so we can log episodic returns
env = gym.wrappers.RecordEpisodeStatistics(env)

# Wrap in SyncVectorEnv so it's compatible with train()
envs = gym.vector.SyncVectorEnv([lambda: env])

In [6]:
# Choo choo train!
agent = train(ppo_config, envs=envs)

SPS: 241
final model saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777666481/balance-bot-ppo_final.cleanrl_model


NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.


Evaluation complete: mean return = nan over 10 episodes


In [7]:
envs.close()

In [10]:
import tensorboardX